In [4]:
!pip install streamlit streamlit-agraph networkx -q

In [5]:
%%writefile app.py
"""🕸️ GraphRAG Visual Editor.

ノード（要素）と関係（エッジ）をブラウザ上で編集し、有向グラフを
ナレッジグラフとして可視化・JSON 出力する Streamlit アプリ。
"""
from __future__ import annotations

import networkx as nx
import streamlit as st
from streamlit_agraph import Config, Edge, Node, agraph

# --- 表示・レイアウト設定（散在する定数の集約） ---
COLOR_DEFAULT = "#F7A7A6"  # 通常ノード（薄いピンク）
COLOR_SOURCE = "#5D5CDE"  # 始点ノード（青紫）
COLOR_TARGET = "#4CAF50"  # 終点ノード（緑）
COLOR_HIGHLIGHT = "#F7A7A6"  # ノードのハイライト色
NODE_SIZE = 25
GRAPH_HEIGHT = 500
EDGE_TYPE = "CURVE_SMOOTH"
COLUMN_RATIO = [3, 1]
SELECTION_KEYS = ("source_node", "target_node")
SELECTION_DEFAULTS = {key: None for key in SELECTION_KEYS}


def init_session_state() -> None:
    """グラフと選択ノードの初期値を、未設定の場合のみ設定する。"""
    if "graph" not in st.session_state:
        st.session_state["graph"] = nx.DiGraph()
    for key, value in SELECTION_DEFAULTS.items():
        if key not in st.session_state:
            st.session_state[key] = value


def clear_selection() -> None:
    """始点・終点の選択を、両方とも解除する。"""
    for key in SELECTION_KEYS:
        st.session_state[key] = None


def add_node(name: str) -> None:
    """入力名のノードをグラフへ追加する。空・重複は警告のみ行う。"""
    if not name:
        st.sidebar.warning("ノード名を入力してください。")
        return
    graph = st.session_state["graph"]
    if graph.has_node(name):
        st.sidebar.warning("そのノードは既に存在します。")
        return
    graph.add_node(name)
    st.sidebar.success(f"追加しました: {name}")


def delete_node(name: str) -> None:
    """指定ノードと、それに接続する全エッジを削除する。

    NetworkX の remove_node は、対象ノードに接続するエッジも自動で除去する。
    削除したノードを始点／終点に選択していた場合は、その選択も解除する。
    """
    graph = st.session_state["graph"]
    if not graph.has_node(name):
        return
    graph.remove_node(name)  # 接続エッジも自動で除去される
    for key in SELECTION_KEYS:
        if st.session_state[key] == name:
            st.session_state[key] = None


def delete_edge(source: str, target: str) -> None:
    """指定した始点→終点のエッジを削除する。存在しない場合は何もしない。"""
    graph = st.session_state["graph"]
    if graph.has_edge(source, target):
        graph.remove_edge(source, target)


def reset_all() -> None:
    """グラフと選択状態を全消去し、画面を再描画する。"""
    st.session_state["graph"].clear()
    clear_selection()
    st.rerun()


def render_sidebar() -> None:
    """サイドバーにノード追加・件数表示・全リセットを描画する。"""
    st.sidebar.header("📦 ノード（要素）の追加")
    new_node = st.sidebar.text_input(
        "新しいノード名を入力", placeholder="新しいノード名"
    )
    if st.sidebar.button("➕ ノードを追加"):
        add_node(new_node)
    st.sidebar.divider()
    graph = st.session_state["graph"]
    st.sidebar.markdown(f"**現在の要素数:** {graph.number_of_nodes()}")
    st.sidebar.markdown(f"**現在の関係数:** {graph.number_of_edges()}")
    if st.sidebar.button("🗑️ 全データをリセット", type="primary"):
        reset_all()


def render_header() -> None:
    """ページタイトルと操作説明を描画する。"""
    st.title("🕸️ GraphRAG Visual Editor")
    st.markdown("ノードをクリックして選択し、関係性を定義してください。")


def node_color(node: str, source: str | None, target: str | None) -> str:
    """ノードの選択状態に応じた表示色を返す。"""
    if node == source:
        return COLOR_SOURCE
    if node == target:
        return COLOR_TARGET
    return COLOR_DEFAULT


def build_display_elements(graph: nx.DiGraph) -> tuple[list[Node], list[Edge]]:
    """グラフから agraph 表示用のノード・エッジのリストを生成する。"""
    source = st.session_state["source_node"]
    target = st.session_state["target_node"]
    nodes = [
        Node(id=n, label=n, size=NODE_SIZE, color=node_color(n, source, target))
        for n in graph.nodes()
    ]
    edges = [
        Edge(source=u, target=v, label=d.get("relation", ""), type=EDGE_TYPE)
        for u, v, d in graph.edges(data=True)
    ]
    return nodes, edges


def graph_config() -> Config:
    """agraph の表示設定（サイズ・物理演算・ハイライト色など）を返す。"""
    return Config(
        width="100%",
        height=GRAPH_HEIGHT,
        directed=True,
        nodeHighlightBehavior=True,
        highlightColor=COLOR_HIGHLIGHT,
        collapsible=False,
        physics=True,
        hierarchical=False,
    )


def render_graph(graph: nx.DiGraph) -> str | None:
    """グラフを描画し、クリックされたノード ID を返す。"""
    nodes, edges = build_display_elements(graph)
    return agraph(nodes=nodes, edges=edges, config=graph_config())


def render_endpoint_buttons(node_id: str) -> None:
    """選択中ノードを始点／終点に設定するボタンを描画する。"""
    st.info(f"選択中: **{node_id}**")
    c1, c2 = st.columns(2)
    with c1:
        if st.button("始点に設定"):
            st.session_state["source_node"] = node_id
            st.rerun()
    with c2:
        if st.button("終点に設定"):
            st.session_state["target_node"] = node_id
            st.rerun()


def render_connection_form(src: str, tgt: str) -> None:
    """始点→終点のエッジ作成フォームを描画する。同一ノードは禁止する。"""
    if src == tgt:
        st.warning("自分自身には接続できません")
        return
    relation_label = st.text_input("関係名 (エッジ名)", key="rel_input")
    if not st.button("🔗 接続する"):
        return
    if not relation_label:
        st.error("関係名を入力してください")
        return
    st.session_state["graph"].add_edge(src, tgt, relation=relation_label)
    clear_selection()
    st.success(f"接続しました: {src} -> {tgt}")
    st.rerun()


def render_clear_button() -> None:
    """始点・終点の選択を解除するボタンを描画する。"""
    if st.button("選択クリア"):
        clear_selection()
        st.rerun()


def render_controls(selected_node_id: str | None) -> None:
    """右カラムに接続操作（始点・終点設定／接続／選択クリア）を描画する。"""
    st.subheader("🛠️ 接続操作")
    if selected_node_id:
        render_endpoint_buttons(selected_node_id)
    else:
        st.write("👈 グラフの丸をクリックしてください")
    st.divider()
    src = st.session_state["source_node"]
    tgt = st.session_state["target_node"]
    st.write(f"**始点 (From):** {src if src else '未選択'}")
    st.write(f"**終点 (To):** {tgt if tgt else '未選択'}")
    if src and tgt:
        render_connection_form(src, tgt)
    if src or tgt:
        render_clear_button()


def edge_label(source: str, target: str, relation: str) -> str:
    """エッジを「始点 --[関係名]--> 終点」の一意な表示文字列にする。

    DiGraph では (始点, 終点) の組が一意なので、この文字列もエッジを一意に表す。
    """
    return f"{source} --[{relation}]--> {target}"


def render_node_delete(node_id: str) -> None:
    """選択中ノードを削除するボタンを描画する。"""
    st.write(f"選択中ノード: **{node_id}**")
    if st.button("🗑️ このノードを削除", key="del_node"):
        delete_node(node_id)
        st.rerun()


def render_edge_delete(graph: nx.DiGraph) -> None:
    """エッジを一覧から選択して削除する UI を描画する。

    agraph のクリックはノード ID しか返さないため、エッジはクリックでは選べない。
    そこで全エッジを一覧化し、選択式で削除する。
    """
    if graph.number_of_edges() == 0:
        st.caption("削除できるエッジはありません。")
        return
    # 表示文字列 -> (始点, 終点) の対応表を作り、選択結果からエッジを特定する
    label_to_edge = {
        edge_label(u, v, d.get("relation", "")): (u, v)
        for u, v, d in graph.edges(data=True)
    }
    choice = st.selectbox(
        "削除するエッジを選択", list(label_to_edge.keys()), key="del_edge_select"
    )
    if st.button("🗑️ このエッジを削除", key="del_edge"):
        u, v = label_to_edge[choice]
        delete_edge(u, v)
        st.rerun()


def render_delete_controls(graph: nx.DiGraph, selected_node_id: str | None) -> None:
    """右カラムに削除操作（ノード削除・エッジ削除）を描画する。"""
    st.subheader("🗑️ 削除操作")
    if selected_node_id:
        render_node_delete(selected_node_id)
    else:
        st.caption("ノード削除: グラフの丸をクリックしてから削除します。")
    render_edge_delete(graph)


def render_graph_json(graph: nx.DiGraph) -> None:
    """グラフデータを JSON 形式で折りたたみ表示する。"""
    with st.expander("📊 生成されたグラフデータ (JSON形式)"):
        st.json(nx.node_link_data(graph))


def main() -> None:
    """アプリ全体を上から順に描画する。"""
    init_session_state()
    render_sidebar()
    render_header()
    graph = st.session_state["graph"]
    col_graph, col_control = st.columns(COLUMN_RATIO)
    with col_graph:
        selected_node_id = render_graph(graph)
    with col_control:
        render_controls(selected_node_id)
        st.divider()
        render_delete_controls(graph, selected_node_id)
    st.divider()
    render_graph_json(graph)


if __name__ == "__main__":
    main()

Overwriting app.py


In [6]:
import subprocess, time, re, os

CF_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
CF_BIN = "/usr/local/bin/cloudflared"

# ① 前回の実行で残ったプロセスを停止（再実行時の "Text file busy" とポート競合を防ぐ）
subprocess.run(["pkill", "-f", "cloudflared"], check=False)  # 該当なしでも止めない
subprocess.run(["pkill", "-f", "streamlit"], check=False)
time.sleep(2)

# ② 一時ファイルに保存してから差し替える（実行中バイナリを直接上書きしない）
tmp_path = CF_BIN + ".download"
result = subprocess.run(
    ["wget", CF_URL, "-O", tmp_path],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError(f"wget 終了コード: {result.returncode}")
os.replace(tmp_path, CF_BIN)   # ファイル名の参照先を入れ替える（上書きではない）
os.chmod(CF_BIN, 0o755)        # 実行権を付与

# ③ Streamlit を起動
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])
time.sleep(5)

# ④ トンネルを開いて公開 URL を表示
cf = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print("URLを取得中...")
for line in cf.stdout:
    text = line.decode("utf-8")
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", text)
    if match:
        print("✅ アクセスURL:", match.group())
        break

URLを取得中...
✅ アクセスURL: https://jesse-affecting-boc-bronze.trycloudflare.com
